In [1]:
%pwd

'/Users/harshpatel/Desktop/Projects/customer-churn-prediction/notebooks'

In [2]:
import os
os.chdir('../')

In [3]:
%pwd

'/Users/harshpatel/Desktop/Projects/customer-churn-prediction'

In [4]:
from pathlib import Path
from dataclasses import dataclass

In [5]:
@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    input_data_path: Path
    base_model_path: Path
    tuned_model_path: Path
    scores_file_path: Path
    base_cm_path: Path
    tuned_cm_path: Path

In [6]:
from CustomerChurnPrediction.constants import *
from CustomerChurnPrediction.utils.common import read_yaml,create_directories

In [7]:
class ConfigurationManager:
    def __init__(self,config_filepath=CONFIG_FILE_PATH):
        self.config = read_yaml(config_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self):

        config = self.config.model_evaluation

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            input_data_path=config.input_data_path,
            base_model_path=config.base_model_path,
            tuned_model_path=config.tuned_model_path,
            scores_file_path=config.scores_file_path,
            base_cm_path = config.base_cm_path,
            tuned_cm_path=config.tuned_cm_path
        )

        return model_evaluation_config

In [8]:
import os
import sys
import json
import joblib
import dagshub
import mlflow
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)
from CustomerChurnPrediction.utils.logger import logger
from CustomerChurnPrediction.utils.exception import CustomException

In [9]:
class ModelEvaluation:
    """Evaluates the base and tuned models on a held-out test set, logs results to MLflow, and saves scores to JSON."""

    THRESHOLD = 0.25

    def __init__(self, config: ModelEvaluationConfig):
        """Sets up config and connects MLflow tracking to DagsHub."""
        self.config = config

        dagshub.init(
            repo_owner=os.getenv("DAGSHUB_REPO_OWNER"),
            repo_name=os.getenv("DAGSHUB_REPO_NAME"),
            mlflow=True,
        )

    def get_input_data(self) -> pd.DataFrame:
        """Loads the transformed data used for training."""
        logger.info(f"Loading data from: {self.config.input_data_path}")
        return pd.read_csv(self.config.input_data_path)

    def split_data(self, df: pd.DataFrame, target_col: str = "Churn", test_size: float = 0.2):
        """Reproduces the same train/test split used during training (same seed)."""
        X = df.drop(columns=[target_col])
        y = df[target_col]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42, stratify=y
        )
        return X_train, X_test, y_train, y_test

    def load_models(self):
        """Loads the saved base and tuned models from disk."""
        base_model = joblib.load(self.config.base_model_path)
        tuned_model = joblib.load(self.config.tuned_model_path)
        logger.info("Loaded base and tuned models")
        return base_model, tuned_model

    def evaluate_model(self, model, X_test, y_test):
        """Computes precision, recall, f1, and roc_auc for a model at the fixed threshold."""
        proba = model.predict_proba(X_test)[:, 1]
        y_pred = (proba >= self.THRESHOLD).astype(int)

        metrics = {
            "precision": precision_score(y_test, y_pred, pos_label=1),
            "recall": recall_score(y_test, y_pred, pos_label=1),
            "f1": f1_score(y_test, y_pred, pos_label=1),
            "roc_auc": roc_auc_score(y_test, proba),
        }

        logger.info(f"Metrics: {metrics}")
        logger.info("\n" + classification_report(y_test, y_pred, digits=3))

        return metrics, y_pred

    def log_confusion_matrix(self, title: str, save_path: str, y_test, y_pred) -> str:
        """Builds a confusion matrix plot for a model and saves it as a PNG."""
        cm = confusion_matrix(y_test, y_pred)

        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.title(f"Confusion Matrix - {title}")

        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path)
        plt.close()

        logger.info(f"Confusion matrix saved to {save_path}")
        return save_path

    def log_to_mlflow(self, run_name: str, model, metrics: dict, cm_path: str) -> None:
        """Logs a model's hyperparameters, evaluation metrics, and confusion matrix to MLflow."""
        with mlflow.start_run(run_name=run_name):
            mlflow.log_params(model.get_params())
            mlflow.log_param("threshold", self.THRESHOLD)
            mlflow.log_metrics(metrics)
            mlflow.log_artifact(cm_path)

        logger.info(f"Logged '{run_name}' evaluation run to MLflow")

    def save_scores(self, scores: dict) -> None:
        """Saves all models' metrics to a single JSON file."""
        os.makedirs(os.path.dirname(self.config.scores_file_path), exist_ok=True)

        with open(self.config.scores_file_path, "w") as f:
            json.dump(scores, f, indent=4)

        logger.info(f"Scores saved to {self.config.scores_file_path}")

    def run_evaluation(self) -> None:
        """Runs the full evaluation pipeline: load data/models, evaluate both, log to MLflow, save scores.json."""
        df = self.get_input_data()
        _, X_test, _, y_test = self.split_data(df)

        base_model, tuned_model = self.load_models()

        base_metrics, base_pred = self.evaluate_model(base_model, X_test, y_test)
        tuned_metrics, tuned_pred = self.evaluate_model(tuned_model, X_test, y_test)

        base_cm_path = self.log_confusion_matrix(
            "XGBoost", self.config.base_cm_path, y_test, base_pred
        )
        tuned_cm_path = self.log_confusion_matrix(
            "XGBoost_Tuned", self.config.tuned_cm_path, y_test, tuned_pred
        )

        # self.log_to_mlflow("XGBoost_Eval", base_model, base_metrics, base_cm_path)
        # self.log_to_mlflow("XGBoost_Tuned_Eval", tuned_model, tuned_metrics, tuned_cm_path)

        scores = {
            "base_model": base_metrics,
            "tuned_model": tuned_metrics,
        }
        self.save_scores(scores)

In [10]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(model_evaluation_config)
    model_evaluation.run_evaluation()
except Exception as e:
    raise CustomException(e,sys)

[2026-06-21 23:56:38,439] 74 CustomerChurnPrediction - INFO - yaml file: config/config.yaml loaded successfully
[2026-06-21 23:56:38,440] 91 CustomerChurnPrediction - INFO - created directory at: artifacts
[2026-06-21 23:56:38,440] 91 CustomerChurnPrediction - INFO - created directory at: artifacts/model_evaluation
[2026-06-21 23:56:45,396] 1025 httpx - INFO - HTTP Request: GET https://dagshub.com/api/v1/user "HTTP/1.1 200 OK"


Accessing as harshpatel16052005

[2026-06-21 23:56:45,413] 107 dagshub - INFO - Accessing as harshpatel16052005
[2026-06-21 23:56:52,377] 1025 httpx - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/harshpatel16052005/customer-churn-prediction "HTTP/1.1 200 OK"
[2026-06-21 23:56:58,482] 1025 httpx - INFO - HTTP Request: GET https://dagshub.com/api/v1/user "HTTP/1.1 200 OK"


Initialized MLflow to track repo "harshpatel16052005/customer-churn-prediction"

[2026-06-21 23:56:58,491] 107 dagshub - INFO - Initialized MLflow to track repo "harshpatel16052005/customer-churn-prediction"


Repository harshpatel16052005/customer-churn-prediction initialized!

[2026-06-21 23:56:58,493] 107 dagshub - INFO - Repository harshpatel16052005/customer-churn-prediction initialized!
[2026-06-21 23:56:58,494] 18 CustomerChurnPrediction - INFO - Loading data from: artifacts/data_transformation/transformed_data.csv
[2026-06-21 23:56:58,592] 35 CustomerChurnPrediction - INFO - Loaded base and tuned models
[2026-06-21 23:56:58,602] 50 CustomerChurnPrediction - INFO - Metrics: {'precision': 0.4538799414348463, 'recall': 0.8288770053475936, 'f1': 0.586565752128666, 'roc_auc': 0.8147276247469859}
[2026-06-21 23:56:58,606] 51 CustomerChurnPrediction - INFO - 
              precision    recall  f1-score   support

           0      0.912     0.639     0.751      1033
           1      0.454     0.829     0.587       374

    accuracy                          0.689      1407
   macro avg      0.683     0.734     0.669      1407
weighted avg      0.790     0.689     0.707      1407

[2026-06-21 23:56:58,613] 50 CustomerChurnPrediction - INFO - Metrics: {'precisi